# 08 · Poster models — TWFE OLS, distance-weighted DiD, and the funding question

This notebook delivers the three pieces agreed with the advisor, using **only** data already in this
research project (`analysis_panel.csv`, `nevi_corridors.geojson`, `ca_counties.csv`):

1. **TWFE OLS linear regression** — the ordinary two-way fixed-effects panel model, exactly as in
   `07_twfe_ols_regression.ipynb`.
2. **Distance-weighted DiD** — same DiD as `03_nevi2022_did_models.ipynb` (controls
   `log_med_hh_inc`, `share_drove_alone` in $X_{ct}$) but now (a) each county is measured by its
   **distance to the NEVI interstate corridor** (the DiD threshold), (b) observations are **weighted
   so that counties closer to the corridor count more** (exponential-decay kernel), and (c) treated vs.
   control counties are separated with an explicit **distance-ring scheme**.
3. **Who funds the corridor?** — a short institutional answer (federal money, *state*-administered, not
   county-chosen) and the DiD is built to be consistent with that fact.

All three share the canonical specification

$$Y_{ct} = \beta\,(\text{Treated}_c \times \text{Post}_t) + X_{ct}\gamma + \alpha_c + \delta_t + \varepsilon_{ct},$$

with county fixed effects $\alpha_c$, year fixed effects $\delta_t$, treatment onset in **2023**, and
standard errors **clustered by county**. $\hat\beta$ on `treat_post` is the DiD estimate.

**Outcomes:** gasoline per-capita (log) and median AQI.

## 0 · Setup and data

In [1]:
import os, glob
from pathlib import Path

# --- Portable paths -------------------------------------------------------------
# Locate the project relative to THIS notebook, so everything keeps working no
# matter where the "Summer Research 2026" folder is placed (or re-downloaded from
# Google Drive) -- as long as the folder structure is kept intact. Expected layout:
#
#   Summer Research 2026/        <- ROOT  (project root; holds the data/ folder)
#   |-- analysis/                <- REV   (analysis_panel.csv, nevi_corridors.geojson, figures/)
#   |   +-- scripts/             <- this notebook lives here
#   +-- data/                    <- county_spatial_data/, air_quality/, ...
#
# No path needs to be edited by hand.
def _find_up(start, needed):
    for d in [start, *start.parents]:
        if all((d / n).exists() for n in needed):
            return d
    return None

HERE = Path.cwd().resolve()

# REV = the analysis/ folder holding the panel + corridor files (search upward first).
REV = _find_up(HERE, ["analysis_panel.csv", "nevi_corridors.geojson"])
if REV is None:                                    # search downward as a fallback
    for base in [HERE, *HERE.parents]:
        hits = list(base.rglob("analysis_panel.csv"))
        if hits:
            REV = hits[0].parent
            break
if REV is None:                                    # last-resort explicit fallback
    REV = Path(r"C:\Users\Trong Khoi Van\Desktop\Denison\Summer Research 2026\analysis")
REV = REV.resolve()

# ROOT = project root (parent of analysis/), which contains the data/ folder.
ROOT = REV.parent
if not (ROOT / "data").exists():                   # robust if analysis/ was nested differently
    up = _find_up(REV, ["data"])
    if up is not None:
        ROOT = up

os.makedirs(REV / "figures", exist_ok=True)
os.chdir(REV)          # analysis_panel.csv, geojson and result CSVs all live here
print("REV  (analysis):", REV)
print("ROOT (project) :", ROOT)


REV  (analysis): C:\Users\Trong Khoi Van\Desktop\Denison\Summer Research 2026\analysis
ROOT (project) : C:\Users\Trong Khoi Van\Desktop\Denison\Summer Research 2026


In [2]:
import json, math
import numpy as np, pandas as pd, warnings
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from linearmodels.panel import PanelOLS
warnings.filterwarnings("ignore")

NAVY, RED, GREY = "#1F3864", "#C0504D", "#BFBFBF"

df = pd.read_csv("analysis_panel.csv", dtype={"fips": str})
df["fips"] = df["fips"].str.zfill(5)

# Controls used in the DiD X_ct vector (per advisor / notebook 03)
CTRL = ["log_med_hh_inc", "share_drove_alone"]

# Gasoline (CEC) ends 2024; AQI series run to 2025
def endyr(o):
    return 2024 if o == "log_gasoline_pc" else 2025

outcomes = {
    "log_gasoline_pc": "Gasoline per-capita (log)",
    "median_aqi":      "Median AQI",
}

print("Panel:", df.shape, "| counties:", df.fips.nunique(),
      "| years:", df.year.min(), "-", df.year.max())
print("Treated (interstate-corridor) counties:", df[df.treated == 1].fips.nunique(),
      "| Control:", df[df.treated == 0].fips.nunique())

Panel: (928, 32) | counties: 58 | years: 2010 - 2025
Treated (interstate-corridor) counties: 23 | Control: 35


### 0.1 · Distance of each county to the NEVI interstate corridor

The DiD "threshold" is the set of NEVI-designated **Interstate** Alternative-Fuel Corridors (the same 12
interstate segments used to assign treatment in `01_corridor_treatment_assignment.ipynb`). For every
county we intersect its polygon with those corridor lines and measure, in kilometres (equirectangular
projection centred on CA),

* `dist_centroid_km` — distance from the county **centroid** to the nearest interstate corridor, and
* `dist_boundary_km` — distance from the county **boundary** (0 if the corridor passes through it).

`dist_centroid_km` is the continuous running variable used for weighting: treated counties (corridor
inside them) sit near 0, and controls fan out to >200 km.

In [3]:
from shapely.geometry import shape
from shapely.ops import unary_union, transform
from shapely import wkt

# --- NEVI interstate corridors (same FIDs as notebook 01) ---
gj = json.load(open("nevi_corridors.geojson"))
feats = gj["features"]
interstate_fids = {4, 8, 12, 21, 22, 27, 31, 33, 34, 36, 37, 42}
iss = unary_union([shape(feats[i - 1]["geometry"]) for i in interstate_fids])

# --- county polygons ---
_cc = Path(ROOT) / "data" / "county_spatial_data" / "ca_counties.csv"
if not _cc.exists():                                # be robust if data/ was moved
    _h = list(Path(ROOT).rglob("ca_counties.csv")) or list(Path(REV).rglob("ca_counties.csv"))
    _cc = _h[0] if _h else _cc
cty = pd.read_csv(_cc, dtype={"GEOID": str})
cty["GEOID"] = cty["GEOID"].str.zfill(5)
cty["geom"] = cty["geometry"].apply(wkt.loads)

lat0 = 37.0
def to_km(x, y, z=None):
    return (x * 111.320 * math.cos(math.radians(lat0)), y * 110.574)

iss_km = transform(to_km, iss)
rows = []
for _, r in cty.iterrows():
    g = transform(to_km, r["geom"])
    rows.append((r["GEOID"], r["NAME"],
                 round(g.centroid.distance(iss_km), 2),
                 round(g.distance(iss_km), 2)))
dist = pd.DataFrame(rows, columns=["fips", "county_name", "dist_centroid_km", "dist_boundary_km"])

df = df.merge(dist[["fips", "dist_centroid_km", "dist_boundary_km"]], on="fips", how="left")
print("Merged distance columns. Range of dist_centroid_km: "
      f"{df.dist_centroid_km.min():.1f} - {df.dist_centroid_km.max():.1f} km")
dist.sort_values("dist_centroid_km").head(10)

Merged distance columns. Range of dist_centroid_km: 1.1 - 247.7 km


,fips,county_name,dist_centroid_km,dist_boundary_km
41,06001,Alameda,1.11,0.0
39,06103,Tehama,1.24,0.0
47,06059,Orange,1.83,0.0
54,06093,Siskiyou,2.62,0.0
27,06113,Yolo,3.85,0.0
42,06057,Nevada,4.97,0.0
5,06037,Los Angeles,5.00,0.0
33,06011,Colusa,6.14,0.0
31,06065,Riverside,6.82,0.0
53,06081,San Mateo,7.00,0.0


## 1 · TWFE OLS linear regression (baseline, per notebook 07)

The county-year data are a panel, so the two-way fixed-effects model is estimated with
`linearmodels.PanelOLS`, which absorbs the county and year fixed effects through the within
transformation (rather than storing dummy columns). `entity_effects=True` = county FE,
`time_effects=True` = year FE, `drop_absorbed=True` drops controls collinear with the FE, and errors
are clustered by county. The coefficient on `treat_post` is the DiD estimate $\hat\beta$.

This is identical in form to `07_twfe_ols_regression.ipynb`; the only change is that the control vector
is the DiD $X_{ct}=\{\texttt{log\_med\_hh\_inc},\ \texttt{share\_drove\_alone}\}$ so that the OLS
baseline and the weighted DiD below use the same covariates and are directly comparable.

In [4]:
def twfe(d, y, xvars, weights=None):
    """Two-way FE panel model (PanelOLS, absorbed FE, county-clustered SE).
    Pass weights=<Series indexed by (fips, year)> for the distance-weighted (WLS) version."""
    d = d.copy().set_index(["fips", "year"])
    kw = dict(entity_effects=True, time_effects=True, drop_absorbed=True)
    if weights is not None:
        kw["weights"] = weights.reindex(d.index)
    m = PanelOLS(d[y], d[xvars], **kw)
    return m.fit(cov_type="clustered", cluster_entity=True)

In [5]:
print("=" * 84)
print("MODEL 1 - TWFE OLS:  Y ~ treat_post + controls | county FE + year FE  (cluster = county)")
print("=" * 84)

ols_rows = []
for o, lab in outcomes.items():
    d = df[df.year <= endyr(o)].dropna(subset=[o] + CTRL).copy()
    r = twfe(d, o, ["treat_post"] + CTRL)
    b, se, p = r.params["treat_post"], r.std_errors["treat_post"], r.pvalues["treat_post"]
    ci = 1.96 * se
    ols_rows.append([lab, b, se, p, ci, int(r.nobs), d.fips.nunique(), r.rsquared_within])
    print(f"\n{lab:26s} beta(treat_post)= {b:+.5f}  SE={se:.5f}  p={p:.3f}  "
          f"N={int(r.nobs)}  R2(within)={r.rsquared_within:.3f}")

ols = pd.DataFrame(ols_rows, columns=["outcome", "beta", "se", "p", "ci95", "N", "n_counties", "r2_within"])
ols.to_csv("did_twfe_ols_results.csv", index=False)
ols[["outcome", "beta", "se", "p", "N", "n_counties", "r2_within"]]

MODEL 1 - TWFE OLS:  Y ~ treat_post + controls | county FE + year FE  (cluster = county)

Gasoline per-capita (log)  beta(treat_post)= -0.04805  SE=0.04098  p=0.241  N=870  R2(within)=0.017

Median AQI                 beta(treat_post)= +0.98126  SE=1.48655  p=0.509  N=844  R2(within)=0.062


,outcome,beta,se,p,N,n_counties,r2_within
0,Gasoline per-capita (log),-0.048046,0.040984,0.241423,870,58,0.017410
1,Median AQI,0.981256,1.486553,0.509394,844,54,0.062082


**Full PanelOLS table** for the headline outcome (edit `PICK` to `median_aqi` for the AQI model):

In [6]:
PICK = "log_gasoline_pc"   # or "median_aqi"
d = df[df.year <= endyr(PICK)].dropna(subset=[PICK] + CTRL).copy()
print(twfe(d, PICK, ["treat_post"] + CTRL))

                          PanelOLS Estimation Summary                           
Dep. Variable:        log_gasoline_pc   R-squared:                        0.0395
Estimator:                   PanelOLS   R-squared (Between):             -0.6859
No. Observations:                 870   R-squared (Within):               0.0174
Date:                Sat, Jul 18 2026   R-squared (Overall):             -0.6854
Time:                        20:02:25   Log-likelihood                    428.76
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      10.898
Entities:                          58   P-value                           0.0000
Avg Obs:                       15.000   Distribution:                   F(3,795)
Min Obs:                       15.000                                           
Max Obs:                       15.000   F-statistic (robust):             4.4921
                            

### 1.1 · TWFE OLS visualizations

In [7]:
# (a) DiD coefficient plot with 95% CI  +  (b) raw treated-vs-control trend for gasoline
fig, axs = plt.subplots(1, 2, figsize=(14, 4.6))

# (a) coefficient plot
ax = axs[0]
yloc = np.arange(len(ols))[::-1]
ax.errorbar(ols.beta, yloc, xerr=ols.ci95, fmt="o", color=NAVY, capsize=4, ms=8, lw=2)
ax.axvline(0, ls="--", c="grey")
ax.set_yticks(yloc); ax.set_yticklabels(ols.outcome)
for yl, b, p in zip(yloc, ols.beta, ols.p):
    ax.annotate(f"{b:+.3f} (p={p:.2f})", (b, yl), textcoords="offset points",
                xytext=(0, 12), ha="center", fontsize=9, color=NAVY)
ax.set_title("TWFE OLS: DiD estimate  beta(treat x post) +/- 95% CI", fontweight="bold")
ax.set_xlabel("Effect of interstate-corridor treatment after 2023")

# (b) raw group means over time (gasoline)
ax = axs[1]
g = (df[df.year <= 2024].dropna(subset=["log_gasoline_pc"])
       .groupby(["year", "treated"]).log_gasoline_pc.mean().unstack())
ax.plot(g.index, g[1], "o-", color=RED, label="Treated (interstate corridor)")
ax.plot(g.index, g[0], "s-", color=NAVY, label="Control (off-interstate)")
ax.axvline(2022.5, ls=":", c="grey"); ax.text(2022.6, ax.get_ylim()[0], " NEVI onset", color="grey")
ax.set_title("Raw trend - gasoline per-capita (log)", fontweight="bold")
ax.set_xlabel("Year"); ax.set_ylabel("mean log gasoline per-capita"); ax.legend(fontsize=9)

fig.tight_layout(); fig.savefig("figures/fig_twfe_ols_summary.png", dpi=150)
print("saved figures/fig_twfe_ols_summary.png"); plt.show()

saved figures/fig_twfe_ols_summary.png


## 2 · Who funds and builds the interstate corridor? (advisor question)

**Short answer: the corridor build-out is _not_ decided or paid for by individual counties. It is a
federal program, administered by the _state_.**

* **Money is federal.** The National Electric Vehicle Infrastructure (NEVI) Formula Program was created
  by the 2021 Bipartisan Infrastructure Law and apportions **\$5 billion (FY2022-2026)** to the states.
  The Federal Highway Administration (FHWA), with the Joint Office of Energy and Transportation,
  administers it.
* **The _state_ decides where it goes.** Each **state DOT** (in California, **Caltrans**) writes the
  annual NEVI plan and must **first build out the FHWA-designated Interstate Alternative-Fuel
  Corridors** - DC fast chargers roughly every 50 miles, within 1 mile of the interstate - before
  spending elsewhere. Counties do **not** nominate, prioritise, or fund these corridor segments.
* **Counties don't self-select.** Which counties are "treated" is therefore fixed by the geography of
  the *federal interstate network* and a *state/federal* build-out rule, not by county wealth, politics,
  or local demand.

**Why this matters for the DiD (and how the model is built to match it).** Because treatment is assigned
by an external federal/state rule rather than chosen by counties, corridor placement is plausibly
**exogenous to county-level unobservables** - exactly the "no self-selection into treatment" condition a
clean DiD needs. The design encodes this in three ways:

1. **Treatment is defined by corridor geography, not county choice** - `treated = 1` iff a
   *federally-designated interstate corridor* physically crosses the county (>=5 km), computed in Sec 2.1.
2. **Standard errors are clustered by county**, because the state assigns the shock at the county level
   and a county's outcomes are correlated across years.
3. **A single common onset year (2023)** - the statewide NEVI build-out turned on at once for every
   corridor county, consistent with one state-level program rather than 58 independent county decisions.
   Year fixed effects absorb any statewide (Caltrans/federal) shock common to all counties.

The distance weighting in Sec 2.3 then leans further on this logic: counties nearer the state-built
corridor are the cleanest comparison, so they receive the most weight.

## 2.1 · Detailed separation of treated vs. control counties

**Treated (23 counties).** A county is treated iff a NEVI **Interstate** corridor physically crosses it
for **>= 5 km** (the 5 km floor drops boundary-clip artefacts such as Napa 1.7 km and Sierra 2.5 km,
following notebook 01). This is a purely geographic, state-determined assignment (see Sec 2).

**Control (35 counties).** Every other California county. Rather than treat all controls as
interchangeable, we grade them by `dist_centroid_km` into transparent **distance rings**, so it is
explicit which controls sit next to the corridor (the credible counterfactual) and which are far
interior/coastal counties:

| Ring | Definition (centroid distance to interstate corridor) | Role |
|------|-------------------------------------------------------|------|
| **Treated** | corridor crosses county (>=5 km) | treatment group |
| **Adjacent control** | 0-25 km | tightest comparison |
| **Near control** | 25-75 km | good comparison |
| **Far control** | > 75 km | weak comparison (heavily down-weighted) |

This is the "detailed way to separate control and treated counties" the advisor asked for; the rings are
also what the distance weights (Sec 2.3) act on.

In [8]:
def ring(row):
    if row.treated == 1:
        return "Treated"
    d = row.dist_centroid_km
    if d <= 25:  return "Adjacent control"
    if d <= 75:  return "Near control"
    return "Far control"

county = (df.drop_duplicates("fips")
            [["fips", "county_name", "treated", "interstate_km",
              "dist_centroid_km", "dist_boundary_km"]].copy())
county["ring"] = county.apply(ring, axis=1)
df = df.merge(county[["fips", "ring"]], on="fips", how="left")

order = ["Treated", "Adjacent control", "Near control", "Far control"]
summary = (county.groupby("ring")
                 .agg(n_counties=("fips", "size"),
                      mean_dist_km=("dist_centroid_km", "mean"),
                      min_dist_km=("dist_centroid_km", "min"),
                      max_dist_km=("dist_centroid_km", "max"))
                 .reindex(order).round(1))
print(summary.to_string())
print("\nAdjacent controls (closest counterfactuals):",
      ", ".join(county[county.ring == "Adjacent control"]
                .sort_values("dist_centroid_km").county_name.tolist()) or "(none within 25 km)")
summary

                  n_counties  mean_dist_km  min_dist_km  max_dist_km
ring                                                                
Treated                 23.0          16.6          1.1        108.6
Adjacent control         NaN           NaN          NaN          NaN
Near control            14.0          48.3         27.8         72.3
Far control             21.0         144.2         77.8        247.7

Adjacent controls (closest counterfactuals): (none within 25 km)


,n_counties,mean_dist_km,min_dist_km,max_dist_km
ring,,,,
Treated,23.0,16.6,1.1,108.6
Adjacent control,NaN,NaN,NaN,NaN
Near control,14.0,48.3,27.8,72.3
Far control,21.0,144.2,77.8,247.7


### 2.2 · Treatment map with distance rings + distance distribution

In [9]:
RING_COL = {"Treated": RED, "Adjacent control": "#E8A33D",
            "Near control": "#7FA6C9", "Far control": GREY}
cmap = dict(zip(county.fips, county.ring))

fig, (axm, axh) = plt.subplots(1, 2, figsize=(14, 8),
                               gridspec_kw={"width_ratios": [1, 1.15]})

# --- map ---
for _, r in cty.iterrows():
    col = RING_COL.get(cmap.get(r["GEOID"].zfill(5), "Far control"), GREY)
    g = r["geom"]; polys = [g] if g.geom_type == "Polygon" else list(g.geoms)
    for poly in polys:
        x, y = poly.exterior.xy; axm.fill(x, y, fc=col, ec="white", lw=.3)
for i, f in enumerate(feats, 1):
    if i in interstate_fids:
        geo = shape(f["geometry"]); lns = [geo] if geo.geom_type == "LineString" else list(geo.geoms)
        for ln in lns:
            x, y = ln.xy; axm.plot(x, y, c=NAVY, lw=1.2)
axm.set_title("Treated vs. control by distance to NEVI interstate corridor", fontweight="bold")
axm.legend(handles=[Patch(fc=RING_COL[k], label=f"{k} (n={int((county.ring==k).sum())})") for k in order]
           + [plt.Line2D([0], [0], color=NAVY, lw=1.5, label="NEVI interstate corridor")],
           loc="lower left", fontsize=8)
axm.set_axis_off(); axm.set_aspect(1.3)

# --- distance histogram ---
axh.hist(county[county.treated == 1].dist_centroid_km, bins=np.arange(0, 260, 15),
         color=RED, alpha=.75, label="Treated")
axh.hist(county[county.treated == 0].dist_centroid_km, bins=np.arange(0, 260, 15),
         color=NAVY, alpha=.55, label="Control")
for x in (25, 75):
    axh.axvline(x, ls=":", c="grey")
axh.set_title("Distance to interstate corridor (county centroid)", fontweight="bold")
axh.set_xlabel("dist_centroid_km"); axh.set_ylabel("number of counties"); axh.legend()

fig.tight_layout(); fig.savefig("figures/fig_distance_rings.png", dpi=150)
print("saved figures/fig_distance_rings.png"); plt.show()

saved figures/fig_distance_rings.png


## 2.3 · Distance weighting - closer counties count more

Each county gets an **exponential-decay weight** on its distance to the interstate corridor,

$$w_c = \exp\!\left(-\,\text{dist\_centroid\_km}_c \,/\, h\right),$$

so a county on the corridor ($\text{dist}\approx0$) has weight ~ 1 and weight falls smoothly with
distance; $h$ (km) is the bandwidth controlling how fast far-away controls fade out. This makes the DiD a
**kernel-weighted / spatial DiD**: identification is driven by the contrast between treated counties and
the *nearby* controls that share the same regional shocks, while distant interior counties barely
enter. We use $h = 50$ km as the headline and report $h \in \{25, 50, 100\}$ for robustness. Weights are
constant within a county over time (a county-level design weight), so they do not interfere with the
within transformation.

In [10]:
def make_weight(d, h):
    """County-level exponential distance weight, indexed by (fips, year), mean-normalised."""
    w = np.exp(-d["dist_centroid_km"] / h)
    w = w / w.mean()
    w.index = pd.MultiIndex.from_frame(d[["fips", "year"]])
    return w

hh = np.linspace(0, 250, 300)
fig, ax = plt.subplots(figsize=(7.5, 4.2))
for h, c in [(25, "#E8A33D"), (50, RED), (100, NAVY)]:
    ax.plot(hh, np.exp(-hh / h), color=c, lw=2, label=f"h = {h} km")
ax.axvspan(0, 25, color="#E8A33D", alpha=.10)
ax.set_title("Exponential-decay distance weight  w = exp(-dist / h)", fontweight="bold")
ax.set_xlabel("distance of county to interstate corridor (km)")
ax.set_ylabel("relative weight"); ax.legend(); ax.set_ylim(0, 1.02)
fig.tight_layout(); fig.savefig("figures/fig_weight_kernel.png", dpi=150)
print("saved figures/fig_weight_kernel.png"); plt.show()

saved figures/fig_weight_kernel.png


## 2.4 · Distance-weighted DiD estimates

Same model as notebook 03 - `Y ~ treat_post + log_med_hh_inc + share_drove_alone | county FE + year FE`,
county-clustered SEs - but fit by **weighted** PanelOLS with the distance weights above. The table shows,
for each outcome, the **unweighted** DiD (all counties equal) next to the **distance-weighted** DiD at
$h=50$ km.

In [11]:
print("=" * 92)
print("MODEL 2 - DiD, unweighted vs. distance-weighted (h = 50 km)")
print("Y ~ treat_post + log_med_hh_inc + share_drove_alone | county FE + year FE (cluster=county)")
print("=" * 92)

H0 = 50
wdid_rows = []
for o, lab in outcomes.items():
    d = df[df.year <= endyr(o)].dropna(subset=[o] + CTRL).copy()
    # unweighted
    r0 = twfe(d, o, ["treat_post"] + CTRL)
    b0, s0, p0 = r0.params["treat_post"], r0.std_errors["treat_post"], r0.pvalues["treat_post"]
    # weighted
    w = make_weight(d, H0)
    r1 = twfe(d, o, ["treat_post"] + CTRL, weights=w)
    b1, s1, p1 = r1.params["treat_post"], r1.std_errors["treat_post"], r1.pvalues["treat_post"]
    wdid_rows.append([lab, b0, s0, p0, b1, s1, p1, int(r1.nobs), d.fips.nunique()])
    print(f"\n{lab:26s}  unweighted b={b0:+.5f} (SE {s0:.5f}, p={p0:.3f})"
          f"   |   weighted b={b1:+.5f} (SE {s1:.5f}, p={p1:.3f})")

wdid = pd.DataFrame(wdid_rows, columns=["outcome", "beta_unw", "se_unw", "p_unw",
                                        "beta_wtd", "se_wtd", "p_wtd", "N", "n_counties"])
wdid.to_csv("did_weighted_results.csv", index=False)
wdid.round(4)

MODEL 2 - DiD, unweighted vs. distance-weighted (h = 50 km)
Y ~ treat_post + log_med_hh_inc + share_drove_alone | county FE + year FE (cluster=county)

Gasoline per-capita (log)   unweighted b=-0.04805 (SE 0.04098, p=0.241)   |   weighted b=-0.06384 (SE 0.05595, p=0.254)

Median AQI                  unweighted b=+0.98126 (SE 1.48655, p=0.509)   |   weighted b=+0.72234 (SE 1.47947, p=0.626)


,outcome,beta_unw,se_unw,p_unw,beta_wtd,se_wtd,p_wtd,N,n_counties
0,Gasoline per-capita (log),-0.0480,0.0410,0.2414,-0.0638,0.0560,0.2543,870,58
1,Median AQI,0.9813,1.4866,0.5094,0.7223,1.4795,0.6255,844,54


**Full weighted PanelOLS table** for the headline outcome:

In [12]:
PICK = "log_gasoline_pc"   # or "median_aqi"
d = df[df.year <= endyr(PICK)].dropna(subset=[PICK] + CTRL).copy()
w = make_weight(d, H0)
print(twfe(d, PICK, ["treat_post"] + CTRL, weights=w))

                          PanelOLS Estimation Summary                           
Dep. Variable:        log_gasoline_pc   R-squared:                        0.0951
Estimator:                   PanelOLS   R-squared (Between):             -1.8337
No. Observations:                 870   R-squared (Within):              -0.1562
Date:                Sat, Jul 18 2026   R-squared (Overall):             -1.8327
Time:                        20:02:42   Log-likelihood                    585.93
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      27.864
Entities:                          58   P-value                           0.0000
Avg Obs:                       15.000   Distribution:                   F(3,795)
Min Obs:                       15.000                                           
Max Obs:                       15.000   F-statistic (robust):             15.029
                            

### 2.5 · Bandwidth robustness

Re-estimate the weighted DiD across bandwidths $h \in \{15, 25, 50, 100, 250\}$ km (larger $h$ closer to
the unweighted estimate). Stable coefficients across $h$ mean the result is not an artefact of one
weighting choice.

In [13]:
bws = [15, 25, 50, 100, 250]
rob_rows = []
for o, lab in outcomes.items():
    d = df[df.year <= endyr(o)].dropna(subset=[o] + CTRL).copy()
    for h in bws:
        w = make_weight(d, h)
        r = twfe(d, o, ["treat_post"] + CTRL, weights=w)
        rob_rows.append([lab, h, r.params["treat_post"], r.std_errors["treat_post"], r.pvalues["treat_post"]])
rob = pd.DataFrame(rob_rows, columns=["outcome", "h", "beta", "se", "p"])
rob.to_csv("did_weighted_bandwidth.csv", index=False)

fig, axs = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, (o, lab) in zip(axs, outcomes.items()):
    s = rob[rob.outcome == lab]
    ax.errorbar(s.h, s.beta, yerr=1.96 * s.se, fmt="o-", color=NAVY, capsize=4)
    ax.axhline(0, ls="--", c="grey")
    ax.set_title(lab, fontweight="bold"); ax.set_xlabel("bandwidth h (km)")
    ax.set_ylabel("beta(treat x post) +/- 95% CI"); ax.set_xscale("log")
    ax.set_xticks(bws); ax.set_xticklabels(bws)
fig.suptitle("Distance-weighted DiD - robustness to weight bandwidth", fontweight="bold")
fig.tight_layout(); fig.savefig("figures/fig_weighted_bandwidth.png", dpi=150)
print("saved figures/fig_weighted_bandwidth.png"); plt.show()

saved figures/fig_weighted_bandwidth.png


### 2.6 · Weighted event study (reference year 2022)

The event study replaces `treat_post` with a full set of treated x year interactions (2022 omitted as the
reference), fit with the same $h=50$ distance weights. Flat, near-zero pre-2023 coefficients support
parallel pre-trends between corridor counties and their nearby controls; movement from 2023 on is the
dynamic treatment effect.

In [14]:
H0 = 50
ev_rows = []
for o, lab in [("log_gasoline_pc", "Gasoline (log)"), ("median_aqi", "Median AQI")]:
    d = df[df.year <= endyr(o)].dropna(subset=[o] + CTRL).copy()
    yrs = [y for y in sorted(d.year.unique()) if y != 2022]
    for y in yrs:
        d[f"D_{y}"] = ((d.year == y) & (d.treated == 1)).astype(float)
    w = make_weight(d, H0)
    r = twfe(d, o, [f"D_{y}" for y in yrs] + CTRL, weights=w)
    for y in sorted(d.year.unique()):
        if y == 2022:
            ev_rows.append([lab, y, 0.0, 0.0, np.nan]); continue
        k = f"D_{y}"
        if k in r.params.index:
            ev_rows.append([lab, y, r.params[k], r.std_errors[k], r.pvalues[k]])
ev = pd.DataFrame(ev_rows, columns=["outcome", "year", "estimate", "se", "p"])
ev.to_csv("did_weighted_event_study.csv", index=False)

fig, axs = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, lab in zip(axs, ["Gasoline (log)", "Median AQI"]):
    s = ev[ev.outcome == lab].sort_values("year")
    ax.axhline(0, ls="--", c="grey"); ax.axvline(2022.5, ls=":", c=RED)
    ax.errorbar(s.year, s.estimate, yerr=1.96 * s.se.fillna(0), fmt="o-", color=NAVY, capsize=3)
    ax.set_title(lab, fontweight="bold"); ax.set_xlabel("Year")
    ax.set_ylabel("Treated x Year (ref 2022)")
fig.suptitle("Distance-weighted DiD event study (h = 50 km, onset 2023)", fontweight="bold")
fig.tight_layout(); fig.savefig("figures/fig_weighted_event_study.png", dpi=150)
print("saved figures/fig_weighted_event_study.png"); plt.show()

saved figures/fig_weighted_event_study.png


## 2.7 · Weighting-scheme comparison — unweighted vs. inverse-distance vs. exponential

Here we add a second, simpler distance weight requested for the poster: an **inverse-distance** weight,

$$w_c = \frac{1}{\text{dist\_centroid\_km}_c},$$

so a county's influence is inversely proportional to how far its centroid sits from the interstate
corridor (mean-normalised, as before). Counties right on the corridor (≈1 km) get the heaviest weight and
weight falls off like $1/d$ — a slower, heavier-tailed decay than the exponential kernel. (No county has
distance 0, so the reciprocal is well defined; a tiny floor is added only as a safeguard.)

We then estimate the **same DiD** three ways and line them up so the poster can show how the estimate
moves as we shift weight toward corridor-adjacent counties:

1. **Unweighted** — every county counts equally (the ordinary TWFE DiD).
2. **Inverse-distance** — $w_c = 1/d$.
3. **Exponential (h = 50 km)** — $w_c = \exp(-d/50)$, the headline kernel from §2.4.

If the three estimates agree in sign and rough magnitude, the effect is robust to how counties are
weighted; if the weighted estimates diverge from the unweighted one, that tells us the corridor-adjacent
counties behave differently from far-away controls.

In [15]:
def make_weight_inv(d):
    """County-level INVERSE-distance weight  w = 1/dist  (mean-normalised),
    indexed by (fips, year). A small floor guards against divide-by-zero."""
    w = 1.0 / d["dist_centroid_km"].clip(lower=0.1)
    w = w / w.mean()
    w.index = pd.MultiIndex.from_frame(d[["fips", "year"]])
    return w

H0 = 50
schemes = {
    "Unweighted":            None,
    "Inverse-distance (1/d)": make_weight_inv,
    "Exponential (h=50)":     (lambda d: make_weight(d, H0)),
}

print("=" * 92)
print("DiD under three weighting schemes:  Y ~ treat_post + controls | county FE + year FE (cluster=county)")
print("=" * 92)

cmp_rows = []
for o, lab in outcomes.items():
    d = df[df.year <= endyr(o)].dropna(subset=[o] + CTRL).copy()
    print(f"\n{lab}")
    for sname, wf in schemes.items():
        w = None if wf is None else wf(d)
        r = twfe(d, o, ["treat_post"] + CTRL, weights=w)
        b, se, p = r.params["treat_post"], r.std_errors["treat_post"], r.pvalues["treat_post"]
        cmp_rows.append([lab, sname, b, se, p, 1.96 * se, int(r.nobs)])
        print(f"   {sname:24s} beta={b:+.5f}  SE={se:.5f}  p={p:.3f}")

cmp = pd.DataFrame(cmp_rows, columns=["outcome", "scheme", "beta", "se", "p", "ci95", "N"])
cmp.to_csv("did_weight_scheme_comparison.csv", index=False)
cmp.round(4)

DiD under three weighting schemes:  Y ~ treat_post + controls | county FE + year FE (cluster=county)

Gasoline per-capita (log)
   Unweighted               beta=-0.04805  SE=0.04098  p=0.241
   Inverse-distance (1/d)   beta=-0.08947  SE=0.05445  p=0.101
   Exponential (h=50)       beta=-0.06384  SE=0.05595  p=0.254

Median AQI
   Unweighted               beta=+0.98126  SE=1.48655  p=0.509
   Inverse-distance (1/d)   beta=-0.08325  SE=1.33028  p=0.950
   Exponential (h=50)       beta=+0.72234  SE=1.47947  p=0.626


,outcome,scheme,beta,se,p,ci95,N
0,Gasoline per-capita (log),Unweighted,-0.0480,0.0410,0.2414,0.0803,870
1,Gasoline per-capita (log),Inverse-distance (1/d),-0.0895,0.0544,0.1008,0.1067,870
2,Gasoline per-capita (log),Exponential (h=50),-0.0638,0.0560,0.2543,0.1097,870
3,Median AQI,Unweighted,0.9813,1.4866,0.5094,2.9136,844
4,Median AQI,Inverse-distance (1/d),-0.0833,1.3303,0.9501,2.6073,844
5,Median AQI,Exponential (h=50),0.7223,1.4795,0.6255,2.8998,844


In [16]:
# Visual comparison: for each outcome, the DiD estimate under each weighting scheme (+/- 95% CI)
scheme_order = ["Unweighted", "Inverse-distance (1/d)", "Exponential (h=50)"]
scheme_col   = {"Unweighted": GREY, "Inverse-distance (1/d)": "#E8A33D", "Exponential (h=50)": RED}

labs = list(outcomes.values())
fig, axs = plt.subplots(1, len(labs), figsize=(6.2 * len(labs), 4.6), squeeze=False)
axs = axs[0]
for ax, lab in zip(axs, labs):
    s = cmp[cmp.outcome == lab].set_index("scheme").reindex(scheme_order)
    x = np.arange(len(scheme_order))
    for xi, sc in zip(x, scheme_order):
        ax.errorbar(xi, s.loc[sc, "beta"], yerr=s.loc[sc, "ci95"],
                    fmt="o", color=scheme_col[sc], capsize=5, ms=10, lw=2)
        ax.annotate(f"{s.loc[sc,'beta']:+.3f}\n(p={s.loc[sc,'p']:.2f})",
                    (xi, s.loc[sc, "beta"]), textcoords="offset points",
                    xytext=(0, 14), ha="center", fontsize=8, color=scheme_col[sc])
    ax.axhline(0, ls="--", c="grey")
    ax.set_xticks(x); ax.set_xticklabels(scheme_order, rotation=15, ha="right")
    ax.set_xlim(-0.5, len(scheme_order) - 0.5)
    ax.set_title(lab, fontweight="bold")
    ax.set_ylabel("beta(treat x post) +/- 95% CI")
fig.suptitle("DiD estimate: unweighted vs. distance-weighted (inverse-distance & exponential)",
             fontweight="bold")
fig.tight_layout(); fig.savefig("figures/fig_weight_scheme_comparison.png", dpi=150)
print("saved figures/fig_weight_scheme_comparison.png"); plt.show()

saved figures/fig_weight_scheme_comparison.png


## 2.8 · Inverse-distance weighted event study (reference year 2022)

The same event study as §2.6, but using the **inverse-distance** weights ($w_c = 1/d$) instead of the
exponential kernel. `treat_post` is replaced by a full set of treated×year interactions (2022 omitted as
the reference). Flat, near-zero pre-2023 coefficients support parallel pre-trends between corridor
counties and their nearer controls; movement from 2023 on is the dynamic treatment effect, now weighted
so that counties closest to the interstate corridor dominate.

In [17]:
ev_inv_rows = []
for o, lab in [("log_gasoline_pc", "Gasoline (log)"), ("median_aqi", "Median AQI")]:
    d = df[df.year <= endyr(o)].dropna(subset=[o] + CTRL).copy()
    yrs = [y for y in sorted(d.year.unique()) if y != 2022]
    for y in yrs:
        d[f"D_{y}"] = ((d.year == y) & (d.treated == 1)).astype(float)
    w = make_weight_inv(d)                          # inverse-distance weights (w = 1/d)
    r = twfe(d, o, [f"D_{y}" for y in yrs] + CTRL, weights=w)
    for y in sorted(d.year.unique()):
        if y == 2022:
            ev_inv_rows.append([lab, y, 0.0, 0.0, np.nan]); continue
        k = f"D_{y}"
        if k in r.params.index:
            ev_inv_rows.append([lab, y, r.params[k], r.std_errors[k], r.pvalues[k]])
ev_inv = pd.DataFrame(ev_inv_rows, columns=["outcome", "year", "estimate", "se", "p"])
ev_inv.to_csv("did_invdist_event_study.csv", index=False)

fig, axs = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, lab in zip(axs, ["Gasoline (log)", "Median AQI"]):
    s = ev_inv[ev_inv.outcome == lab].sort_values("year")
    ax.axhline(0, ls="--", c="grey"); ax.axvline(2022.5, ls=":", c=RED)
    ax.errorbar(s.year, s.estimate, yerr=1.96 * s.se.fillna(0), fmt="o-", color=NAVY, capsize=3)
    ax.set_title(lab, fontweight="bold"); ax.set_xlabel("Year")
    ax.set_ylabel("Treated x Year (ref 2022)")
fig.suptitle("Inverse-distance-weighted DiD event study (w = 1/d, onset 2023)", fontweight="bold")
fig.tight_layout(); fig.savefig("figures/fig_invdist_event_study.png", dpi=150)
print("saved figures/fig_invdist_event_study.png"); plt.show()

saved figures/fig_invdist_event_study.png


## 3 · Poster summary table

In [18]:
poster = wdid[["outcome", "beta_unw", "se_unw", "p_unw", "beta_wtd", "se_wtd", "p_wtd", "N", "n_counties"]].copy()
poster.columns = ["Outcome", "TWFE OLS b", "SE", "p", "Weighted DiD b (h=50)", "SE ", "p ", "N", "Counties"]
poster = poster.round({"TWFE OLS b": 4, "SE": 4, "p": 3, "Weighted DiD b (h=50)": 4, "SE ": 4, "p ": 3})
poster.to_csv("poster_summary.csv", index=False)
print("Saved: did_twfe_ols_results.csv, did_weighted_results.csv, did_weighted_bandwidth.csv,")
print("       did_weighted_event_study.csv, poster_summary.csv")
print("Figures in ./figures/: fig_twfe_ols_summary, fig_distance_rings, fig_weight_kernel,")
print("       fig_weighted_bandwidth, fig_weighted_event_study (.png)")
poster

Saved: did_twfe_ols_results.csv, did_weighted_results.csv, did_weighted_bandwidth.csv,
       did_weighted_event_study.csv, poster_summary.csv
Figures in ./figures/: fig_twfe_ols_summary, fig_distance_rings, fig_weight_kernel,
       fig_weighted_bandwidth, fig_weighted_event_study (.png)


,Outcome,TWFE OLS b,SE,p,Weighted DiD b (h=50),SE,p,N,Counties
0,Gasoline per-capita (log),-0.0480,0.0410,0.241,-0.0638,0.0560,0.254,870,58
1,Median AQI,0.9813,1.4866,0.509,0.7223,1.4795,0.626,844,54


### How to read this for the poster

* **TWFE OLS b** = the ordinary two-way FE DiD estimate (notebook 07 spec) - every county counts equally.
* **Weighted DiD b** = the same DiD but identification concentrated on counties *near the state-built
  interstate corridor*, via exponential distance weights (closer = heavier). Agreement between the two
  columns says the effect is not driven by far-away, poorly-comparable counties.
* **Funding note for the poster:** the interstate corridor is a **federally funded, state (Caltrans)-
  administered** build-out - counties neither choose nor pay for it - which is why corridor placement can
  be treated as an external shock and the DiD is credible.

*Sign/magnitude interpretation:* for `log_gasoline_pc`, b is roughly a proportional (~ percent) change in
per-capita gasoline for corridor counties after 2023; for median AQI, b is in AQI points.